# Build an MCP Server -- tool logic only (Colab/Kaggle)

**Scope of this notebook: narrow, on purpose.** The [main lesson](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/mcp-server) explains why Colab and Kaggle are *not* a good fit for the actual MCP server project -- an MCP server is a standalone local process that a desktop AI client (Claude Desktop) launches and talks to directly over stdio, and neither Colab nor Kaggle can offer that. That's still true, and this notebook does not attempt to work around it.

What this notebook *does* let you do: experiment with the two **tool functions themselves** -- `search_course_topics` and `count_words` -- as plain Python, with no `@mcp.tool()` decorator, no `FastMCP` server object, and no MCP protocol involved at all. This is useful for understanding and tweaking the *logic* the real server wraps, in a free hosted notebook, before (or instead of) setting anything up locally.

This is **not** a substitute for the real project. To get an actual MCP server running and connected to Claude Desktop, follow the [full lesson](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/mcp-server) locally with `uv`.

## Get some real course content to search

The real `search_course_topics` tool (in [`examples/mcp-server/server.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/mcp-server/server.py)) searches this course's actual `docs/` folder. To keep that same behavior here, we do a shallow clone of the course repo -- just the latest commit, not its full history -- so there's real lesson content on disk to search over.

In [ ]:
!git clone --depth 1 https://github.com/abderrahim-lectures/python-data-analysis-course.git /tmp/repo

## The tool functions, as plain Python

Same logic as `server.py` in the course repo, minus the `@mcp.tool()` decorator and the `FastMCP` server wrapper -- these are just ordinary functions here, callable directly, with no protocol layer between you and the code.

In [ ]:
from pathlib import Path

DOCS_DIR = Path("/tmp/repo/docs")


def search_course_topics(query: str) -> str:
    """Search this course's real lesson files for a topic and report which pages mention it.

    Looks through every .md/.mdx file under docs/ for `query` (case-insensitive)
    and returns each matching file's path plus one line of surrounding context.
    This is the same logic the real MCP tool runs -- just called directly here,
    instead of through the MCP protocol.
    """
    if not DOCS_DIR.exists():
        return f"docs/ not found at {DOCS_DIR} -- did the git clone cell run successfully?"

    query_lower = query.lower()
    matches = []
    for path in sorted(DOCS_DIR.rglob("*.md*")):
        text = path.read_text(encoding="utf-8", errors="ignore")
        for line in text.splitlines():
            if query_lower in line.lower():
                snippet = line.strip()[:160]
                matches.append(f'{path.relative_to(DOCS_DIR.parent)}: "{snippet}"')
                break  # one hit per file is enough context
        if len(matches) >= 5:
            break

    if not matches:
        return f"No lesson pages mention '{query}'."
    return "Found in:\n" + "\n".join(matches)


def count_words(text: str) -> int:
    """Count the words in a piece of text, splitting on whitespace."""
    return len(text.split())

## Try `search_course_topics`

Call it exactly like a normal Python function -- no server, no client, no protocol.

In [ ]:
print(search_course_topics("groupby"))

In [ ]:
print(search_course_topics("MCP"))

## Try `count_words`

In [ ]:
print(count_words("The Model Context Protocol lets an AI assistant call code that lives outside of it."))

In [ ]:
print(count_words("one two three four five"))

## What's missing here -- and where to get it

This notebook only ever calls these two functions directly, in the same Python process, the same way you'd call any other function. It does **not**:

- register them as MCP tools with `@mcp.tool()` and a `FastMCP` server object,
- run a persistent local server process over stdio,
- or connect anything to Claude Desktop (or any other MCP client).

That's the actual point of the project -- a real, standalone server any MCP-compatible client can plug into. For that, follow the [Build an MCP Server lesson](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/mcp-server) locally with `uv`, using [`examples/mcp-server/server.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/mcp-server/server.py) (which wraps these exact same functions with `@mcp.tool()`) as your starting point.